# Project 2: Fertilizer Optimization Intelligence  
## AI-Driven Precision Agriculture and Resource Optimization

**AAI-510 Final Team Project**  
**Module focus:** Fertilizer optimization, soil health intelligence, sustainability, and resource readiness

---

## Project Purpose

This notebook supports the second workstream in the integrated precision agriculture project.

Project 1 answers:

> **What crop should a farmer plant?**

Project 2 answers:

> **How should the farmer manage nutrients, soil health, and resource readiness to support that crop?**

The goal is not to duplicate the crop recommendation work. This notebook builds the next decision-support layer by turning soil nutrient and environmental data into fertilizer and soil-health intelligence.

## Business Problem

Farmers often make fertilizer decisions using experience, fixed schedules, or general recommendations. That approach can create four practical problems:

- under-fertilization, which can limit crop readiness
- over-fertilization, which increases cost and may damage soil or water quality
- poor nutrient balance across Nitrogen, Phosphorus, and Potassium
- weak operational readiness before planting

This project uses machine learning and transparent scoring logic to support better fertilizer planning, soil-health awareness, and resource readiness.

### Business Question

**How can soil nutrient and environmental data be used to recommend fertilizer actions and soil-health priorities before planting?**

## Relationship to the Overall Project

This notebook is designed to consolidate with Andrea's crop recommendation work as one final project.

| Workstream | Main Question | Primary Output |
|---|---|---|
| Project 1: Intelligent Crop Recommendation | What crop should be planted? | Recommended crop / crop suitability |
| Project 2: Fertilizer Optimization Intelligence | How should the soil and nutrients be managed? | Fertilizer readiness / soil-health recommendation |

Integrated decision flow:

```text
Soil + Environmental Inputs
        ↓
Project 1: Crop Recommendation
        ↓
Project 2: Fertilizer and Soil-Health Intelligence
        ↓
Farmer-Facing Recommendation
```

## Dataset Overview

The project uses the **Crop Recommendation Dataset** from Kaggle.

Dataset source: https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset

| Feature | Meaning | Project 2 relevance |
|---|---|---|
| N | Nitrogen level | Fertilizer and nutrient balance |
| P | Phosphorus level | Fertilizer and nutrient balance |
| K | Potassium level | Fertilizer and nutrient balance |
| temperature | Environmental temperature | Heat/resource readiness |
| humidity | Relative humidity | Water/resource readiness |
| ph | Soil pH | Nutrient availability and soil suitability |
| rainfall | Rainfall level | Irrigation/resource readiness |
| label | Crop label | Crop-aware nutrient baseline |

### Scope Boundary

The dataset does **not** include actual yield, fertilizer application quantities, farm financials, irrigation schedules, soil type, location, or seasonality. Therefore, this notebook does **not** claim exact yield prediction or certified fertilizer dosage. Outputs are framed as **decision-support indicators**.

## Notebook Workflow

1. Load and validate the dataset  
2. Review data quality  
3. Perform fertilizer-focused EDA  
4. Use existing engineered features when available  
5. Add Project 2 intelligence features without duplicating Andrea's core feature engineering  
6. Create fertilizer readiness and action categories  
7. Train and compare ML models  
8. Evaluate model performance  
9. Explain model behavior  
10. Provide business recommendations, risks, and deployment considerations

In [ ]:
# Core libraries
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Persistence
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Dataset

The notebook first looks for Andrea's engineered dataset. If it is unavailable, it falls back to the cleaned or raw Kaggle dataset so the notebook remains reproducible.

Expected possible paths:

```text
data/processed/engineered_ag_data.csv
data/processed/crop_recommendation_clean.csv
data/raw/Crop_recommendation.csv
Crop_recommendation.csv
```

In [ ]:
candidate_paths = [
    Path("data/processed/engineered_ag_data.csv"),
    Path("data/processed/crop_recommendation_clean.csv"),
    Path("data/raw/Crop_recommendation.csv"),
    Path("Crop_recommendation.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place Crop_recommendation.csv in data/raw/ "
        "or provide engineered_ag_data.csv in data/processed/."
    )

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Shape: {df.shape}")
df.head()

## 2. Standardize Columns and Validate Required Fields

This keeps the notebook stable if the raw and processed versions use slightly different column capitalization.

In [ ]:
df.columns = [col.strip().lower() for col in df.columns]

required_cols = ["n", "p", "k", "temperature", "humidity", "ph", "rainfall", "label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Rows before duplicate removal: {before}")
print(f"Rows after duplicate removal: {after}")
df.info()

## 3. Data Quality Review

This section confirms whether the dataset is usable for fertilizer and soil-health intelligence.

In [ ]:
missing_summary = df[required_cols].isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary

In [ ]:
numeric_cols = ["n", "p", "k", "temperature", "humidity", "ph", "rainfall"]
df[numeric_cols].describe().T

In [ ]:
crop_counts = df["label"].value_counts().sort_values(ascending=False)
crop_counts

### Data Quality Interpretation

The dataset contains the core signals needed for this project: NPK nutrient levels, pH, rainfall, humidity, temperature, and crop labels. These variables are enough to build decision-support indicators for nutrient balance, soil health, and resource readiness. The dataset is not enough for certified agronomic prescriptions, so the final recommendations are intentionally framed as guidance categories.

## 4. Fertilizer-Focused EDA

This EDA is focused on fertilizer optimization and soil-health intelligence, not crop classification. The goal is to understand how nutrient and environmental conditions vary across crop labels and how that variation can support operational recommendations.

In [ ]:
crop_npk_profile = (
    df.groupby("label")[["n", "p", "k"]]
    .mean()
    .round(2)
    .sort_index()
)
crop_npk_profile.head()

In [ ]:
ax = crop_npk_profile.plot(kind="bar", figsize=(14, 6))
ax.set_title("Average NPK Levels by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Nutrient Level")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

### NPK Interpretation

Average nutrient levels differ across crops. This supports a crop-aware fertilizer strategy instead of one generic recommendation. In a real farming workflow, the crop recommendation from Project 1 should influence the nutrient-readiness guidance from Project 2.

In [ ]:
crop_ph_profile = (
    df.groupby("label")["ph"]
    .agg(["mean", "min", "max"])
    .round(2)
    .sort_values("mean")
)
crop_ph_profile.head(10)

In [ ]:
ax = crop_ph_profile["mean"].plot(kind="bar", figsize=(14, 5))
ax.set_title("Average Soil pH by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average pH")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

### pH Interpretation

Soil pH matters because it affects nutrient availability. Even when NPK levels appear sufficient, poor pH conditions can reduce the usefulness of those nutrients. This makes pH a relevant input for soil-health scoring.

In [ ]:
crop_water_profile = (
    df.groupby("label")[["rainfall", "humidity", "temperature"]]
    .mean()
    .round(2)
    .sort_values("rainfall", ascending=False)
)
crop_water_profile.head()

In [ ]:
ax = crop_water_profile[["rainfall", "humidity"]].plot(kind="bar", figsize=(14, 6))
ax.set_title("Average Rainfall and Humidity by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Value")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

### Water and Resource Interpretation

Rainfall and humidity create useful context for irrigation and resource readiness. Project 2 uses these environmental variables as support signals, not as direct irrigation scheduling prescriptions.

In [ ]:
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix for Soil and Environmental Variables")
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im)
plt.tight_layout()
plt.show()

### Correlation Interpretation

The correlation matrix helps identify whether variables move together or provide independent signals. Weak correlations can still be useful because they may capture different aspects of soil condition, weather context, and crop suitability.

## 5. Integration With Andrea's Feature Engineering

Andrea's feature engineering work already covers foundational transformations such as nutrient ratios, total NPK, water availability, heat-humidity effects, pH categories, and nutrient deficiency features.

To avoid duplicate ownership, this notebook does **not** position those as the main contribution. Instead, it uses them when available and creates lightweight fallback versions only when needed for standalone execution.

Project 2 contribution begins at the intelligence layer:

- Soil Health Score
- Fertilizer Readiness Category
- Sustainability Score
- Resource Readiness Category
- Fertilizer Action Category
- ML model for fertilizer action prediction

In [ ]:
df_fe = df.copy()
EPS = 1e-6

# Compatibility fallback only: create base engineered features if Andrea's processed file is not present.
# These are not treated as the unique Project 2 contribution.
if "np_ratio" not in df_fe.columns:
    df_fe["np_ratio"] = df_fe["n"] / (df_fe["p"] + EPS)
if "nk_ratio" not in df_fe.columns:
    df_fe["nk_ratio"] = df_fe["n"] / (df_fe["k"] + EPS)
if "pk_ratio" not in df_fe.columns:
    df_fe["pk_ratio"] = df_fe["p"] / (df_fe["k"] + EPS)
if "total_npk" not in df_fe.columns:
    df_fe["total_npk"] = df_fe["n"] + df_fe["p"] + df_fe["k"]

if "water_availability_index" not in df_fe.columns:
    df_fe["water_availability_index"] = (
        0.6 * df_fe["rainfall"].rank(pct=True) +
        0.4 * df_fe["humidity"].rank(pct=True)
    ) * 100

if "heat_stress_index" not in df_fe.columns:
    temp_rank = df_fe["temperature"].rank(pct=True)
    humidity_inverse = 1 - df_fe["humidity"].rank(pct=True)
    df_fe["heat_stress_index"] = (0.7 * temp_rank + 0.3 * humidity_inverse) * 100

if "ph_distance_from_neutral" not in df_fe.columns:
    df_fe["ph_distance_from_neutral"] = (df_fe["ph"] - 7.0).abs()
if "ph_suitability_score" not in df_fe.columns:
    df_fe["ph_suitability_score"] = (100 - (df_fe["ph_distance_from_neutral"] * 20)).clip(0, 100)

base_engineered_cols = [
    "np_ratio", "nk_ratio", "pk_ratio", "total_npk", "water_availability_index",
    "heat_stress_index", "ph_distance_from_neutral", "ph_suitability_score"
]

df_fe[base_engineered_cols].head()

## 6. Crop-Aware Nutrient Baselines

The dataset does not provide official fertilizer prescriptions. To stay honest, this notebook estimates crop-aware nutrient baselines from observed crop-level averages in the dataset. These are used to create decision-support categories, not exact agronomic prescriptions.

In [ ]:
baseline_cols = {"ideal_n", "ideal_p", "ideal_k"}

if not baseline_cols.issubset(df_fe.columns):
    nutrient_baselines = (
        df_fe.groupby("label")[["n", "p", "k"]]
        .mean()
        .rename(columns={"n": "ideal_n", "p": "ideal_p", "k": "ideal_k"})
    )
    df_fe = df_fe.merge(nutrient_baselines, on="label", how="left")

for nutrient in ["n", "p", "k"]:
    ideal_col = f"ideal_{nutrient}"
    gap_col = f"{nutrient}_gap"
    pct_gap_col = f"{nutrient}_pct_gap"
    if gap_col not in df_fe.columns:
        df_fe[gap_col] = df_fe[ideal_col] - df_fe[nutrient]
    if pct_gap_col not in df_fe.columns:
        df_fe[pct_gap_col] = df_fe[gap_col] / (df_fe[ideal_col] + EPS)

if "total_abs_nutrient_gap" not in df_fe.columns:
    df_fe["total_abs_nutrient_gap"] = df_fe[["n_gap", "p_gap", "k_gap"]].abs().sum(axis=1)

df_fe[["label", "n", "ideal_n", "n_gap", "p", "ideal_p", "p_gap", "k", "ideal_k", "k_gap"]].head()

## 7. Fertilizer Action Labels

The target variable for Project 2 is created from nutrient gap logic. This is a practical way to convert raw and engineered soil indicators into an operational recommendation category.

The categories are decision-support labels:

- increase_N
- increase_P
- increase_K
- rebalance_NPK
- maintain_current_levels

In [ ]:
def nutrient_status(pct_gap, low_threshold=0.15, high_threshold=-0.15):
    if pct_gap > low_threshold:
        return "low"
    if pct_gap < high_threshold:
        return "excess"
    return "balanced"

for nutrient in ["n", "p", "k"]:
    df_fe[f"{nutrient}_status"] = df_fe[f"{nutrient}_pct_gap"].apply(nutrient_status)


def fertilizer_action(row):
    gaps = {
        "increase_N": row["n_pct_gap"],
        "increase_P": row["p_pct_gap"],
        "increase_K": row["k_pct_gap"],
    }
    positive_gaps = {k: v for k, v in gaps.items() if v > 0.15}
    excess_count = sum(row[f"{n}_status"] == "excess" for n in ["n", "p", "k"])

    if len(positive_gaps) == 0 and excess_count == 0:
        return "maintain_current_levels"
    if len(positive_gaps) >= 2 or excess_count >= 2:
        return "rebalance_NPK"
    if len(positive_gaps) == 1:
        return max(positive_gaps, key=positive_gaps.get)
    return "rebalance_NPK"

if "fertilizer_action" not in df_fe.columns:
    df_fe["fertilizer_action"] = df_fe.apply(fertilizer_action, axis=1)

df_fe["fertilizer_action"].value_counts()

## 8. Project 2 Intelligence Scores

This section adds the unique Project 2 layer. These scores convert nutrient and environmental signals into business-readable outputs.

The scores are intentionally transparent. They are not certified agronomic formulas; they are ML project decision-support indicators.

In [ ]:
# Nutrient balance score: higher is better, based on lower total nutrient gaps.
max_gap = df_fe["total_abs_nutrient_gap"].max()
df_fe["nutrient_balance_score"] = (100 - (df_fe["total_abs_nutrient_gap"] / (max_gap + EPS) * 100)).clip(0, 100)

# Soil health score combines nutrient balance and pH suitability.
df_fe["soil_health_score"] = (
    0.65 * df_fe["nutrient_balance_score"] +
    0.35 * df_fe["ph_suitability_score"]
).clip(0, 100)

# Resource readiness combines soil health with water availability and heat stress.
df_fe["resource_readiness_score"] = (
    0.45 * df_fe["soil_health_score"] +
    0.35 * df_fe["water_availability_index"] +
    0.20 * (100 - df_fe["heat_stress_index"])
).clip(0, 100)

# Sustainability score penalizes large nutrient imbalance and excess nutrient status.
excess_count = (
    (df_fe["n_status"] == "excess").astype(int) +
    (df_fe["p_status"] == "excess").astype(int) +
    (df_fe["k_status"] == "excess").astype(int)
)
df_fe["sustainability_score"] = (
    0.70 * df_fe["nutrient_balance_score"] +
    0.30 * df_fe["ph_suitability_score"] -
    10 * excess_count
).clip(0, 100)


def score_category(score):
    if score >= 75:
        return "High"
    if score >= 50:
        return "Moderate"
    return "Low"


def readiness_category(row):
    if row["soil_health_score"] >= 75 and row["fertilizer_action"] == "maintain_current_levels":
        return "High Readiness"
    if row["soil_health_score"] >= 50:
        return "Moderate Readiness"
    return "Needs Intervention"


df_fe["soil_health_category"] = df_fe["soil_health_score"].apply(score_category)
df_fe["resource_readiness_category"] = df_fe["resource_readiness_score"].apply(score_category)
df_fe["sustainability_category"] = df_fe["sustainability_score"].apply(score_category)
df_fe["fertilizer_readiness_category"] = df_fe.apply(readiness_category, axis=1)

df_fe[[
    "soil_health_score", "soil_health_category", "resource_readiness_score",
    "resource_readiness_category", "sustainability_score", "sustainability_category",
    "fertilizer_readiness_category", "fertilizer_action"
]].head()

In [ ]:
summary_counts = pd.DataFrame({
    "soil_health_category": df_fe["soil_health_category"].value_counts(),
}).fillna(0)
summary_counts

In [ ]:
fertilizer_summary = df_fe["fertilizer_action"].value_counts().to_frame("count")
fertilizer_summary["pct"] = (fertilizer_summary["count"] / len(df_fe) * 100).round(2)
fertilizer_summary

### Intelligence Layer Interpretation

The soil health, sustainability, and resource readiness scores make the model outputs more usable for business and operational decisions. Instead of only showing a crop label or a raw nutrient number, the system gives farmers a practical readiness signal and a fertilizer action category.

## 9. Feature Selection

The rubric requires feature selection and explanation. For Project 2, candidate features are selected based on three criteria:

1. direct relevance to fertilizer action prediction
2. operational interpretability
3. availability at decision time

The final feature set includes raw soil/environment inputs, compatibility fallback features, crop-aware nutrient gaps, and Project 2 intelligence scores.

In [ ]:
candidate_feature_cols = [
    "n", "p", "k", "temperature", "humidity", "ph", "rainfall",
    "np_ratio", "nk_ratio", "pk_ratio", "total_npk",
    "water_availability_index", "heat_stress_index", "ph_distance_from_neutral", "ph_suitability_score",
    "n_gap", "p_gap", "k_gap", "n_pct_gap", "p_pct_gap", "k_pct_gap",
    "total_abs_nutrient_gap", "nutrient_balance_score", "soil_health_score",
    "resource_readiness_score", "sustainability_score"
]

feature_cols = [col for col in candidate_feature_cols if col in df_fe.columns]

X = df_fe[feature_cols].copy()
y = df_fe["fertilizer_action"].copy()

print(f"Selected {len(feature_cols)} features")
feature_cols

In [ ]:
# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

feature_target_corr = X.copy()
feature_target_corr["target_encoded"] = y_encoded
corr_to_target = (
    feature_target_corr.corr(numeric_only=True)["target_encoded"]
    .drop("target_encoded")
    .abs()
    .sort_values(ascending=False)
    .to_frame("abs_corr_to_target")
)

corr_to_target.head(12)

### Feature Selection Interpretation

The selected features are not chosen only because they improve model performance. They are also useful because they can be explained to farming stakeholders: nutrient gaps, pH suitability, resource readiness, and sustainability all map back to real decisions.

## 10. Modeling Objective

The supervised learning task is to predict the **fertilizer action category**.

This supports the operational question:

> Based on soil nutrients and environmental conditions, what fertilizer action category should be recommended?

The notebook compares a baseline model with tree-based models to satisfy model comparison and evaluation requirements.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Target classes:", list(label_encoder.classes_))

## 11. Train and Compare Models

The model comparison includes a simple baseline and stronger nonlinear models.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average="weighted", zero_division=0
    )
    results.append({
        "model": name,
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    })

results_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False).reset_index(drop=True)
results_df

In [ ]:
ax = results_df.set_index("model")[["accuracy", "f1_weighted"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Model Comparison: Fertilizer Action Prediction")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Model Comparison Interpretation

The final model should be selected based on performance and usability. For this project, a strong model is not only accurate; it must also be explainable enough to support trust in a farming decision-support setting.

## 12. Final Model Evaluation

This section evaluates the best-performing model using classification metrics and a confusion matrix.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]
y_pred = best_model.predict(X_test)

print(f"Best model selected: {best_model_name}")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm)
ax.set_title(f"Confusion Matrix - {best_model_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks(np.arange(len(label_encoder.classes_)))
ax.set_yticks(np.arange(len(label_encoder.classes_)))
ax.set_xticklabels(label_encoder.classes_, rotation=45, ha="right")
ax.set_yticklabels(label_encoder.classes_)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar(im)
plt.tight_layout()
plt.show()

### Evaluation Interpretation

Accuracy and F1-score show how consistently the model predicts fertilizer action categories. In business terms, false recommendations matter because they can lead to unnecessary fertilizer use or missed nutrient interventions. This is why human review and local agronomic calibration remain important before real deployment.

## 13. Explainability

Feature importance is used to explain which nutrient and environmental factors drive the fertilizer action model. This supports trust and makes the results easier to present to both technical and nontechnical audiences.

In [ ]:
def get_feature_importance(model, feature_names):
    estimator = model.named_steps["model"] if hasattr(model, "named_steps") else model

    if hasattr(estimator, "feature_importances_"):
        importance = estimator.feature_importances_
    elif hasattr(estimator, "coef_"):
        importance = np.abs(estimator.coef_).mean(axis=0)
    else:
        return None

    return (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

importance_df = get_feature_importance(best_model, feature_cols)
importance_df.head(15) if importance_df is not None else "Feature importance not available for selected model."

In [ ]:
if importance_df is not None:
    ax = importance_df.head(15).sort_values("importance").plot(
        kind="barh",
        x="feature",
        y="importance",
        legend=False,
        figsize=(10, 6)
    )
    ax.set_title(f"Top Feature Drivers - {best_model_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

### Explainability Interpretation

The most important features should be reviewed for business plausibility. If nutrient gaps, soil health, pH suitability, or NPK-related variables appear near the top, the model is aligned with the intended fertilizer optimization use case.

## 14. Optional SHAP Explainability

This section is optional. If SHAP is installed, it can provide additional explanation for model predictions. If SHAP is not installed, the notebook still satisfies the explainability requirement through model feature importance.

In [ ]:
try:
    import shap

    estimator = best_model.named_steps["model"] if hasattr(best_model, "named_steps") else best_model
    X_sample = X_test.sample(min(100, len(X_test)), random_state=RANDOM_STATE)

    if best_model_name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
        explainer = shap.Explainer(estimator, X_train)
        shap_values = explainer(X_sample)
        shap.plots.beeswarm(shap_values, max_display=12, show=True)
    else:
        print("SHAP skipped for this selected model type. Feature importance is used instead.")
except Exception as e:
    print("SHAP not available or not compatible in this environment.")
    print(f"Reason: {e}")

## 15. Farmer-Facing Recommendation Function

The model output should be translated into plain language. This function converts the predicted fertilizer action into a practical decision-support message.

In [ ]:
def generate_fertilizer_guidance(row, model=best_model):
    input_df = pd.DataFrame([row])[feature_cols]
    pred_encoded = model.predict(input_df)[0]
    pred_action = label_encoder.inverse_transform([pred_encoded])[0]

    guidance_map = {
        "increase_N": "Increase nitrogen support based on the crop-aware nutrient gap.",
        "increase_P": "Increase phosphorus support based on the crop-aware nutrient gap.",
        "increase_K": "Increase potassium support based on the crop-aware nutrient gap.",
        "rebalance_NPK": "Review the overall NPK balance before planting or fertilizing.",
        "maintain_current_levels": "Current nutrient balance appears acceptable for this decision-support framework.",
    }

    return {
        "crop": row.get("label"),
        "predicted_fertilizer_action": pred_action,
        "guidance": guidance_map.get(pred_action, "Review nutrient conditions."),
        "soil_health_score": round(row.get("soil_health_score", np.nan), 2),
        "resource_readiness_score": round(row.get("resource_readiness_score", np.nan), 2),
        "sustainability_score": round(row.get("sustainability_score", np.nan), 2),
    }

In [ ]:
sample_outputs = []
for _, row in df_fe.sample(min(10, len(df_fe)), random_state=RANDOM_STATE).iterrows():
    result = generate_fertilizer_guidance(row.to_dict())
    result.update({
        "N_status": row["n_status"],
        "P_status": row["p_status"],
        "K_status": row["k_status"],
    })
    sample_outputs.append(result)

recommendation_table = pd.DataFrame(sample_outputs)
recommendation_table

## 16. Save Project 2 Outputs

Saving the engineered dataset and model artifacts supports reproducibility and a realistic deployment discussion. These files should be committed only if the team decides generated artifacts belong in the repository.

In [ ]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(parents=True, exist_ok=True)

engineered_output_path = Path("data/processed/fertilizer_intelligence_features.csv")
model_output_path = Path("models/fertilizer_action_model.joblib")
encoder_output_path = Path("models/fertilizer_action_label_encoder.joblib")
features_output_path = Path("models/fertilizer_action_features.joblib")

df_fe.to_csv(engineered_output_path, index=False)
joblib.dump(best_model, model_output_path)
joblib.dump(label_encoder, encoder_output_path)
joblib.dump(feature_cols, features_output_path)

print(f"Saved engineered features to: {engineered_output_path}")
print(f"Saved model to: {model_output_path}")
print(f"Saved label encoder to: {encoder_output_path}")
print(f"Saved feature list to: {features_output_path}")

## 17. Deployment Strategy

A full application is not required for this course, but the deployment discussion must be realistic.

### Proposed Workflow

```text
Farmer / Soil Sensor Input
        ↓
Data Validation
        ↓
Project 1: Crop Recommendation
        ↓
Project 2: Fertilizer Optimization Intelligence
        ↓
Soil Health + Resource Readiness Scores
        ↓
Farmer Recommendation Dashboard or Mobile App
```

### Batch vs. Real-Time

- **Batch use case:** seasonal planning before planting.
- **Near-real-time use case:** updated recommendations when new soil or weather readings are available.

### Monitoring Needs

- input data quality checks
- seasonal drift monitoring
- local soil calibration
- recommendation outcome tracking
- periodic retraining when new regional data becomes available

## 18. Ethical, Business, and Environmental Risks

### Risk 1: Overconfidence

The system should not replace local agronomists or farmer expertise. It should support decision-making.

### Risk 2: Dataset Generalization

The dataset may not represent every region, soil type, crop variety, or season. Real deployment would require local calibration.

### Risk 3: Over-Fertilization

Incorrect recommendations could increase fertilizer waste or environmental runoff. Human review and conservative recommendation thresholds are needed.

### Risk 4: Missing Yield and Cost Data

The dataset does not include actual yield or financial results, so the project cannot prove direct profit improvement.

### Mitigation

Frame outputs as decision-support indicators, validate locally, monitor outcomes, and involve agricultural experts before production use.

## 19. Business Recommendations

A practical implementation should:

1. Use Project 1 crop recommendation and Project 2 fertilizer intelligence together.  
2. Start as a decision-support tool for crop planning and soil preparation.  
3. Provide recommendation categories instead of exact fertilizer quantities.  
4. Add local agronomic calibration before real deployment.  
5. Track outcomes over time to improve the recommendation framework.  
6. Integrate with soil testing services, weather feeds, or IoT sensors when available.

## 20. Conclusion

This notebook completes the second workstream of the integrated precision agriculture project.

The main Project 2 contribution is the **Fertilizer Optimization Intelligence** layer, which provides:

- crop-aware nutrient gap analysis
- soil health scoring
- sustainability scoring
- resource readiness scoring
- fertilizer action classification
- model comparison and evaluation
- explainable recommendations
- deployment and risk discussion

Together with Project 1, this creates a stronger final project story:

> **Recommend what to plant, then recommend how to manage nutrients and resources to support that crop.**

## Rubric Alignment Checklist

| Rubric Requirement | Where it is addressed |
|---|---|
| Problem statement and justification | Business Problem and Project Purpose |
| Data understanding / EDA | Data Quality Review and Fertilizer-Focused EDA |
| Data preparation / feature engineering | Integration with Andrea's Feature Engineering and Project 2 Intelligence Scores |
| Feature selection | Feature Selection section |
| Modeling | Model training and comparison |
| Evaluation | Final Model Evaluation and confusion matrix |
| Deployment | Deployment Strategy section |
| Discussion and conclusions | Business Recommendations, Risks, and Conclusion |

This notebook is intentionally scoped as a graduate-level ML decision-support workflow, not a production agronomy tool.